In [108]:
import numpy as np
import tensorflow as tf
from tf_pwa.config_loader import ConfigLoader 
import json
import os

In [109]:
# --- Configuration ---
CONFIG_FILE = "config_a.yml"
PARAMS_FILE = "final_params_full.json"

# Ensure we are in the correct directory (Analysis)
if not os.path.exists(CONFIG_FILE):
    print("Warning: config file not found in current directory.")

In [110]:
# 1. Load Configuration
config = ConfigLoader(CONFIG_FILE)

# 2. Load Parameters (to maintain system properties)
with open(PARAMS_FILE, 'r') as f:
    params_dict = json.load(f)['value']

/home/akazatsky/Desktop/CompProject/New/B2DxDK/tf-pwa/tf_pwa/amp/core.py:138: UserWarning: No model named C(BWR_LS) found, use default instead.
  warnings.warn(
/home/akazatsky/Desktop/CompProject/New/B2DxDK/tf-pwa/tf_pwa/amp/core.py:138: UserWarning: No model named C(BWR) found, use default instead.
  warnings.warn(
/home/akazatsky/Desktop/CompProject/New/B2DxDK/tf-pwa/tf_pwa/amp/core.py:138: UserWarning: No model named C(New) found, use default instead.
  warnings.warn(
/home/akazatsky/Desktop/CompProject/New/B2DxDK/tf-pwa/tf_pwa/amp/core.py:138: UserWarning: No model named C(one) found, use default instead.
  warnings.warn(
/home/akazatsky/Desktop/CompProject/New/B2DxDK/tf-pwa/tf_pwa/amp/core.py:138: UserWarning: No model named C2(BWR) found, use default instead.
  warnings.warn(


In [111]:
# 3. Setup Kinematics (Hardcoded from tf_pwa_analysis_Gemini.py)
particles = list(config.get_decay().outs)
particle_map = {p.name: p for p in particles}

# Vectors matching Julia comparison
p4_dict = {
    particle_map["D"]: tf.constant([[2.0452, -0.1467, 0.2235, -0.7847]], dtype=tf.float64),
    particle_map["D0"]: tf.constant([[2.2606, 0.2284, -0.3689, 1.2019]], dtype=tf.float64),
    particle_map["K"]: tf.constant([[0.7718, -0.0873, 0.1803, -0.5584]], dtype=tf.float64),
    particle_map["pi"]: tf.constant([[0.2017, 0.0056, -0.0349, 0.1413]], dtype=tf.float64)
}

phsp_variables = config.data.cal_angle(p4_dict)
phsp_variables["c"] = np.array([-1.0]) # Extra variable

In [112]:
# 4. Set Parameters for Psi(4040)
# Isolate Psi(4040) (Chain Index 5)
# Chain: Bp -> Psi(4040) + K+; Psi(4040) -> D+ Dst0 (D0 pi0) .. wait, D+ Dst-?
# B -> D K Dx

amp_model = config.get_amplitude()
dg = amp_model.decay_group
all_chains = dg.chains

chain_idx = 5
chain = all_chains[chain_idx]

print(f"Selected Chain: {chain}")

# Set used chains to ONLY this one
dg.set_used_chains([chain_idx])

# Prepare Parameters: All zero except this chain's couplings = 1.0
p_new = params_dict.copy()

# Reset all couplings to 0
for k in p_new:
    if "total" in k or "g_ls" in k:
        if k.endswith("r") or k.endswith("i"):
            p_new[k] = 0.0

# Set Psi(4040) couplings to 1.0
# Iterate over step in chain to build prefix
for d_idx, d in enumerate(chain.chain):
    c_name = d.core.name.replace("(1+)", "(1.)")
    o_names = [p.name.replace("(1+)", "(1.)") for p in d.outs]
    prefix = f"{c_name}->{'.'.join(o_names)}"
    
    # Prod LS=0 (idx=0), Decay LS=0 (idx=0) for Psi(4040) [L=1, l=1]??
    # Wait, Psi(4040) is [L=1, l=1].
    # In tf_pwa_analysis_Gemini.py lines 80:
    # ("Psi(4040) [L=1, l=1]", 5, 0, 0)
    # So Prod_LS index is 0. Decay_LS index is 0.
    # Let's verify ls list length for Psi(4040).
    
    ls_idx = 0
    # Hardcoded 0 based on granular_groups definition
    
    for k in p_new:
        if prefix in k and ("total" in k or "g_ls" in k):
            if k.endswith(f"_{ls_idx}r"):
                print(f"Setting {k} = 1.0")
                p_new[k] = 1.0

config.set_params(p_new)


Selected Chain: [Bp->Psi(4040)+K, Psi(4040)->Dst+D, Dst->D0+pi]
Setting Bp->Psi(4040).KPsi(4040)->Dst.DDst->D0.pi_total_0r = 1.0
Setting Bp->Psi(4040).K_g_ls_0r = 1.0
Setting Bp->Psi(4040).KPsi(4040)->Dst.DDst->D0.pi_total_0r = 1.0
Setting Psi(4040)->Dst.D_g_ls_0r = 1.0
Setting Bp->X(3872).KX(3872)->Dst.DDst->D0.pi_total_0r = 1.0
Setting Dst->D0.pi_g_ls_0r = 1.0
Setting Bp->X(3915)(0-).KX(3915)(0-)->Dst.DDst->D0.pi_total_0r = 1.0
Setting Bp->chi(c2)(3930).Kchi(c2)(3930)->Dst.DDst->D0.pi_total_0r = 1.0
Setting Bp->X(3940)(1.).KX(3940)(1.)->Dst.DDst->D0.pi_total_0r = 1.0
Setting Bp->X(3993).KX(3993)->Dst.DDst->D0.pi_total_0r = 1.0
Setting Bp->Psi(4040).KPsi(4040)->Dst.DDst->D0.pi_total_0r = 1.0
Setting Bp->X(4300).KX(4300)->Dst.DDst->D0.pi_total_0r = 1.0
Setting Bp->NR(0-)SPp.KNR(0-)SPp->Dst.DDst->D0.pi_total_0r = 1.0
Setting Bp->NR(1.)PSp.KNR(1.)PSp->Dst.DDst->D0.pi_total_0r = 1.0
Setting Bp->NR(0-)SPm.KNR(0-)SPm->Dst.DDst->D0.pi_total_0r = 1.0
Setting Bp->NR(1-)PPm.KNR(1-)PPm->Dst.DDst

/home/akazatsky/Desktop/CompProject/New/B2DxDK/tf-pwa/tf_pwa/variable.py:578: UserWarning: X(3872)_theta0 not found
  warnings.warn("{} not found".format(name))
/home/akazatsky/Desktop/CompProject/New/B2DxDK/tf-pwa/tf_pwa/variable.py:578: UserWarning: X(3940)(1.)_theta0 not found
  warnings.warn("{} not found".format(name))
/home/akazatsky/Desktop/CompProject/New/B2DxDK/tf-pwa/tf_pwa/variable.py:578: UserWarning: X(3993)_theta0 not found
  warnings.warn("{} not found".format(name))
/home/akazatsky/Desktop/CompProject/New/B2DxDK/tf-pwa/tf_pwa/variable.py:578: UserWarning: X(4300)_theta0 not found
  warnings.warn("{} not found".format(name))
/home/akazatsky/Desktop/CompProject/New/B2DxDK/tf-pwa/tf_pwa/variable.py:578: UserWarning: NR(0-)SPp_alpha not found
  warnings.warn("{} not found".format(name))
/home/akazatsky/Desktop/CompProject/New/B2DxDK/tf-pwa/tf_pwa/variable.py:578: UserWarning: NR(0-)SPp_beta not found
  warnings.warn("{} not found".format(name))


True

In [113]:
# 5. Calculate Amplitude
val = dg.get_amp(phsp_variables).numpy().flatten()[0]

print(f"\nCalculated Amplitude for Psi(4040):")
print(f"{val}\n")
print(f"Real: {val.real}")
print(f"Imag: {val.imag}")
print(f"Abs:  {abs(val)}")


Calculated Amplitude for Psi(4040):
(-0.0006049977354135942-0.0030870270680673287j)

Real: -0.0006049977354135942
Imag: -0.0030870270680673287
Abs:  0.0031457524344480677


In [114]:
#-0.0006049977354135942 - 0.0030870270680673287im

In [115]:
# --- D-Matrix, H-Factor, Angle, and Lineshape Inspection ---
print("\n--- Inspecting D-Matrices, H-Factors, Angles, and Lineshapes ---\n")

# We use a monkey-patching approach to intercept the calculation

saved_d_matrices = []
saved_h_factors = []
saved_r_values = []

# 1. Identify Decay Class (for D and H)
DecayClass = type(dg.chains[0][0])
original_get_D = DecayClass.get_D_matrix_term
original_get_H = DecayClass.get_helicity_amp

# 2. Identify Resonance Classes (for Lineshape R)
# Collect all unique resonance classes from the inner particles of the first chain
resonance_classes = set()
original_get_amp_res_map = {}

for particle in dg.chains[0].inner:
    res_class = type(particle)
    if res_class not in resonance_classes:
        resonance_classes.add(res_class)
        original_get_amp_res_map[res_class] = res_class.get_amp

print(f"Identified {len(resonance_classes)} resonance classes to patch: {[c.__name__ for c in resonance_classes]}")

# --- Interception Functions ---

def intercepted_get_D(self_decay, data, data_p, **kwargs):
    ret = original_get_D(self_decay, data, data_p, **kwargs)
    if len(saved_d_matrices) < 50:
        val = ret.numpy()[0]
        
        # Extract angles
        try:
            b_particle = self_decay.outs[0]
            ang = data[b_particle]["ang"]
            
            if isinstance(ang, dict):
                ang_val = {}
                for k, v in ang.items():
                    if hasattr(v, 'numpy'):
                        ang_val[k] = v.numpy()[0]
                    else:
                        ang_val[k] = v
            elif hasattr(ang, 'numpy'):
                ang_val = ang.numpy()[0]
            else:
                ang_val = str(ang)
        except Exception as e:
            ang_val = f"Error: {e}"
            
        saved_d_matrices.append({
            "name": str(self_decay),
            "value": val,
            "shape": ret.shape,
            "angles": ang_val
        })
    return ret

def intercepted_get_H(self_decay, data, data_p, **kwargs):
    ret = original_get_H(self_decay, data, data_p, **kwargs)
    if len(saved_h_factors) < 50:
        val = ret.numpy()[0]
        saved_h_factors.append({
            "name": str(self_decay),
            "value": val,
            "shape": ret.shape
        })
    return ret

def make_intercepted_get_amp_res(original_func):
    def intercepted(self_res, *args, **kwargs):
        ret = original_func(self_res, *args, **kwargs)
        if len(saved_r_values) < 50:
            # Capture
            val = ret
            shape = "scalar"
            if hasattr(ret, "numpy"):
                val = ret.numpy()[0]
                shape = ret.shape
            
            saved_r_values.append({
                "name": str(self_res),
                "value": val,
                "shape": shape
            })
        return ret
    return intercepted

# --- Apply Patches ---
DecayClass.get_D_matrix_term = intercepted_get_D
DecayClass.get_helicity_amp = intercepted_get_H

for res_class in resonance_classes:
    res_class.get_amp = make_intercepted_get_amp_res(original_get_amp_res_map[res_class])

print("Running amplitude calculation...")
try:
    dg.get_amp(phsp_variables)
except Exception as e:
    print(f"Calculation error: {e}")
finally:
    # Restore original methods
    DecayClass.get_D_matrix_term = original_get_D
    DecayClass.get_helicity_amp = original_get_H
    for res_class, orig_func in original_get_amp_res_map.items():
        res_class.get_amp = orig_func

# --- Print Results ---

print(f"\nCaptured {len(saved_d_matrices)} D-matrices.")
print(f"Captured {len(saved_h_factors)} H-factors.")
print(f"Captured {len(saved_r_values)} R-values (Lineshapes).\n")

print("=== D-Matrices and Angles ===")
for i, item in enumerate(saved_d_matrices):
    print(f"Decay [{i}]: {item['name']}")
    print(f"  Shape: {item['shape']}")
    print(f"  Angles: {item['angles']}")
    print(f"  Value (idx 0):\n{item['value']}")
    print("-" * 30)

print("\n=== H-Factors ===")
for i, item in enumerate(saved_h_factors):
    print(f"Decay [{i}]: {item['name']}")
    print(f"  Shape: {item['shape']}")
    print(f"  Value (idx 0):\n{item['value']}")
    print("-" * 30)

print("\n=== Lineshape Values (R) ===")
for i, item in enumerate(saved_r_values):
    print(f"Particle [{i}]: {item['name']}")
    print(f"  Shape: {item['shape']}")
    print(f"  Value (idx 0):\n{item['value']}")
    print("-" * 30)



--- Inspecting D-Matrices, H-Factors, Angles, and Lineshapes ---

Identified 2 resonance classes to patch: ['ParticleOne', 'Particle']
Running amplitude calculation...

Captured 3 D-matrices.
Captured 3 H-factors.
Captured 2 R-values (Lineshapes).

=== D-Matrices and Angles ===
Decay [0]: Bp->Psi(4040)+K
  Shape: (1, 1, 3, 1)
  Angles: {'alpha': np.float64(-1.1198740857780907), 'beta': np.float64(0.3444357722368102), 'gamma': np.float64(0.0), 'D_matrix_0': array([[1.+0.j]])}
  Value (idx 0):
[[[0.+0.j]
  [1.+0.j]
  [0.+0.j]]]
------------------------------
Decay [1]: Psi(4040)->Dst+D
  Shape: (1, 3, 3, 1)
  Angles: {'alpha': np.float64(1.9905579342206075), 'beta': np.float64(0.03425037969726589), 'gamma': np.float64(0.0), 'D_matrix_2': array([[-4.07423259e-01-9.12918336e-01j, -9.86821650e-03-2.21118348e-02j,
        -1.19509251e-04-2.67785857e-04j],
       [-2.42139409e-02+0.00000000e+00j,  9.99413513e-01+0.00000000e+00j,
         2.42139409e-02+0.00000000e+00j],
       [-1.19509251e-

In [116]:
print(-4.07423259e-01-9.12918336e-01j)
print(-9.86821650e-03-2.21118348e-02j)
print(-1.19509251e-04-2.67785857e-04j)

print(-2.42139409e-02+0.00000000e+00j)
print(9.99413513e-01+0.00000000e+00j)
print(2.42139409e-02+0.00000000e+00j)

print(-1.19509251e-04+2.67785857e-04j)
print(9.86821650e-03-2.21118348e-02j)
print(-4.07423259e-01+9.12918336e-01j)

print()
print(0.12378949+0.33405639j)
print(-0.86380842+0.j)
print(-0.12378949+0.33405639j)

(-0.407423259-0.912918336j)
(-0.0098682165-0.0221118348j)
(-0.000119509251-0.000267785857j)
(-0.0242139409+0j)
(0.999413513+0j)
(0.0242139409+0j)
(-0.000119509251+0.000267785857j)
(0.0098682165-0.0221118348j)
(-0.407423259+0.912918336j)

(0.12378949+0.33405639j)
(-0.86380842+0j)
(-0.12378949+0.33405639j)


In [117]:
D_lprim_0 = np.array([0.12378949+0.33405639j, -0.86380842+0.j, -0.12378949+0.33405639j])

D_l_lprim = np.array([[-4.07423259e-01-9.12918336e-01j, -9.86821650e-03-2.21118348e-02j, -1.19509251e-04-2.67785857e-04j],
                     [2.42139409e-02+0.00000000e+00j, 9.99413513e-01+0.00000000e+00j, -2.42139409e-02+0.00000000e+00j],
                     [-1.19509251e-04+2.67785857e-04j, 9.86821650e-03-2.21118348e-02j, 4.07423259e-01-9.12918336e-01j]]).reshape(3,3)

D_0_l = np.array([0+0j, 1+0j, 0+0j])

H_0_l = np.array([0+0j, -0.91871445+0j, 0+0j])
H_l_lprim = np.array([0.7876652+0j, 0+0j, -0.7876652+0.j])

R_Psi = -0.26369565350533064+0.051679259589068605j

# Calculate A matrices (element-wise multiplication)
A_0_l = D_0_l * H_0_l
A_l_lprim = D_l_lprim * H_l_lprim

print("A_0_l:")
print(A_0_l)
print()
print("A_l_lprim:")
print(A_l_lprim)


A_0_l:
[ 0.        +0.j -0.91871445+0.j  0.        +0.j]

A_l_lprim:
[[-3.20913123e-01-7.19074004e-01j  0.00000000e+00-0.00000000e+00j
   9.41332781e-05+2.10925601e-04j]
 [ 1.90724786e-02+0.00000000e+00j  0.00000000e+00+0.00000000e+00j
   1.90724786e-02-0.00000000e+00j]
 [-9.41332781e-05+2.10925601e-04j  0.00000000e+00+0.00000000e+00j
  -3.20913123e-01+7.19074004e-01j]]


In [118]:
print(A_0_l * R_Psi @ A_l_lprim @ D_lprim_0.transpose())
print(-0.0006049977354135942 - 0.0030870270680673287j)

(0.0006049977307115193+0.0030870270440747894j)
(-0.0006049977354135942-0.0030870270680673287j)
